# 01 — Label Taxonomy (T130)

Defines the 5-label router taxonomy for the Concierge message intent classifier
and maps the public CLINC150 dataset onto those labels.

**Dataset**: [CLINC150](https://github.com/clinc/oos-eval) via `datasets.load_dataset("clinc_oos", "plus")`.
CLINC150 has 150 in-scope intents across 10 domains plus an out-of-scope (oos) class.
We collapse a subset of those 151 intents into our 5 router labels.

**Wire contract**: `specs/001-concierge-platform/contracts/internal/modelserver.yaml`
fixes the label enum at exactly five strings; do not rename them without amending the contract.

Downstream notebooks (02 TF-IDF, 03 small DL/ONNX, 04 LLM zero-shot, 05 compare/export)
all read from the CSV produced here.

## Labels

| Label | Definition | Mapped from CLINC150 | Agent routes to |
|-------|-----------|---------------------|-----------------|
| `spam` | Out-of-scope / off-topic for the tenant | `oos` class | no tool, polite non-engagement |
| `faq` | Visitor wants information | general-knowledge / informational intents | `rag_search` over CMS content |
| `lead_intent` | Visitor signals contact / booking / sales | contact, scheduling, sales-adjacent intents | `capture_lead` |
| `escalate` | Complaint, cancellation, account issue | complaint / cancellation intents | `escalate` (human handoff) |
| `ambiguous` | Short / conversational, could be multiple intents | small-talk and meta intents | ask one clarifying question |

In [1]:
from __future__ import annotations

import hashlib
from pathlib import Path

import pandas as pd
from datasets import load_dataset

OUR_LABELS: tuple[str, ...] = ("spam", "faq", "lead_intent", "escalate", "ambiguous")

In [2]:
# (1) Load CLINC150 plus config.
# Legacy `clinc_oos` ID was script-loaded; modern huggingface_hub requires
# `namespace/name`. The maintained mirror lives at `clinc/clinc_oos`.
ds = load_dataset("clinc/clinc_oos", "plus")
print(ds)

README.md:   0%|          | 0.00/24.0k [00:00<?, ?B/s]

C:\Users\ahmad\OneDrive\Desktop\AIE bootcamp\week8_project\CONCIERGE\backend\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ahmad\.cache\huggingface\hub\datasets--clinc--clinc_oos. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plus/train-00000-of-00001.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

plus/validation-00000-of-00001.parquet:   0%|          | 0.00/77.8k [00:00<?, ?B/s]

plus/test-00000-of-00001.parquet:   0%|          | 0.00/136k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15250 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'intent'],
        num_rows: 15250
    })
    validation: Dataset({
        features: ['text', 'intent'],
        num_rows: 3100
    })
    test: Dataset({
        features: ['text', 'intent'],
        num_rows: 5500
    })
})


In [3]:
# (2) Print the full intent list with indices
intent_feature = ds["train"].features["intent"]
ALL_INTENTS: list[str] = list(intent_feature.names)
print(f"Total intents (incl. oos): {len(ALL_INTENTS)}\n")
for i, name in enumerate(ALL_INTENTS):
    print(f"{i:3d}  {name}")

Total intents (incl. oos): 151

  0  restaurant_reviews
  1  nutrition_info
  2  account_blocked
  3  oil_change_how
  4  time
  5  weather
  6  redeem_rewards
  7  interest_rate
  8  gas_type
  9  accept_reservations
 10  smart_home
 11  user_name
 12  report_lost_card
 13  repeat
 14  whisper_mode
 15  what_are_your_hobbies
 16  order
 17  jump_start
 18  schedule_meeting
 19  meeting_schedule
 20  freeze_account
 21  what_song
 22  meaning_of_life
 23  restaurant_reservation
 24  traffic
 25  make_call
 26  text
 27  bill_balance
 28  improve_credit_score
 29  change_language
 30  no
 31  measurement_conversion
 32  timer
 33  flip_coin
 34  do_you_have_pets
 35  balance
 36  tell_joke
 37  last_maintenance
 38  exchange_rate
 39  uber
 40  car_rental
 41  credit_limit
 42  oos
 43  shopping_list
 44  expiration_date
 45  routing
 46  meal_suggestion
 47  tire_change
 48  todo_list
 49  card_declined
 50  rewards_balance
 51  change_accent
 52  vaccines
 53  reminder_update
 54  foo

In [4]:
# (3) Mapping from CLINC150 intent strings to our 5 labels.
# Curated to cover representative intents per category. Intents NOT in this dict
# are filtered out so the classifier trains on a clean projection of the taxonomy.
INTENT_TO_LABEL: dict[str, str] = {
    # spam <- oos
    "oos": "spam",
    # faq <- informational / general knowledge
    "tell_joke": "faq",
    "fun_fact": "faq",
    "definition": "faq",
    "meaning_of_life": "faq",
    "weather": "faq",
    "time": "faq",
    "date": "faq",
    "current_location": "faq",
    "what_is_your_name": "faq",
    "who_made_you": "faq",
    "calories": "faq",
    "nutrition_info": "faq",
    "recipe": "faq",
    "restaurant_suggestion": "faq",
    "ingredients_list": "faq",
    # lead_intent <- contact / scheduling / sales-adjacent
    "make_call": "lead_intent",
    "schedule_meeting": "lead_intent",
    "schedule_maintenance": "lead_intent",
    "reminder": "lead_intent",
    "calendar": "lead_intent",
    "calendar_update": "lead_intent",
    "book_flight": "lead_intent",
    "book_hotel": "lead_intent",
    "restaurant_reservation": "lead_intent",
    "meeting_schedule": "lead_intent",
    "order": "lead_intent",
    "order_status": "lead_intent",
    # escalate <- cancellations / account issues
    "cancel": "escalate",
    "cancel_reservation": "escalate",
    "report_fraud": "escalate",
    "report_lost_card": "escalate",
    "freeze_account": "escalate",
    "replacement_card_duration": "escalate",
    "card_declined": "escalate",
    "damaged_card": "escalate",
    "expiration_date": "escalate",
    # ambiguous <- small talk / meta / boundary
    "greeting": "ambiguous",
    "goodbye": "ambiguous",
    "thank_you": "ambiguous",
    "you_are_welcome": "ambiguous",
    "yes": "ambiguous",
    "no": "ambiguous",
    "maybe": "ambiguous",
    "what_can_i_ask_you": "ambiguous",
}

print(f"Mapping covers {len(INTENT_TO_LABEL)} intents -> {len(set(INTENT_TO_LABEL.values()))} labels.")
for lbl in OUR_LABELS:
    count = sum(1 for v in INTENT_TO_LABEL.values() if v == lbl)
    print(f"  {lbl:12s} <- {count} intents")

Mapping covers 45 intents -> 5 labels.
  spam         <- 1 intents
  faq          <- 15 intents
  lead_intent  <- 12 intents
  escalate     <- 9 intents
  ambiguous    <- 8 intents


In [5]:
# Sanity check: every mapped intent must exist in the CLINC150 intent list.
# Unknown entries are dropped from the working mapping with a warning so the
# pipeline does not silently train on a typo.
unknown = [intent for intent in INTENT_TO_LABEL if intent not in ALL_INTENTS]
if unknown:
    print(f"WARNING: {len(unknown)} mapped intents not found in CLINC150 and will be skipped:")
    for u in unknown:
        print(f"  - {u}")
    INTENT_TO_LABEL = {k: v for k, v in INTENT_TO_LABEL.items() if k in ALL_INTENTS}
else:
    print(f"OK: all {len(INTENT_TO_LABEL)} mapped intents exist in CLINC150.")

  - you_are_welcome


In [6]:
# (4) Filter to mapped intents and (5) apply the mapping.
SPLIT_RENAME = {"train": "train", "validation": "val", "test": "test"}

frames: list[pd.DataFrame] = []
for split_name in ds.keys():
    df = ds[split_name].to_pandas()
    df["original_intent"] = df["intent"].map(lambda i: ALL_INTENTS[i])
    df = df[df["original_intent"].isin(INTENT_TO_LABEL)].copy()
    df["label"] = df["original_intent"].map(INTENT_TO_LABEL)
    df["split"] = SPLIT_RENAME.get(split_name, split_name)
    frames.append(df[["text", "label", "original_intent", "split"]])

mapped = pd.concat(frames, ignore_index=True)
print(f"Total mapped rows: {len(mapped)}")
mapped.head()

Total mapped rows: 7800


,text,label,original_intent,split
0,what is the meaning of realism,faq,definition,train
1,what is regard mean,faq,definition,train
2,what is the meaning of interorganizational,faq,definition,train
3,what is it is all relative mean,faq,definition,train
4,what is intercontinental mean,faq,definition,train


In [7]:
# Class distribution across splits.
dist = (
    mapped.groupby(["label", "split"]).size().unstack(fill_value=0).reindex(list(OUR_LABELS))
)
dist["total"] = dist.sum(axis=1)
print(dist)
print(f"\nGrand total: {len(mapped)} rows")

split        test  train  val  total
label                               
spam         1000    250  100   1350
faq           450   1500  300   2250
lead_intent   360   1200  240   1800
escalate      270    900  180   1350
ambiguous     210    700  140   1050

Grand total: 7800 rows


In [8]:
# (6) Save the mapped dataset to CSV.
OUT = Path("data/clinc150_mapped.csv")
OUT.parent.mkdir(parents=True, exist_ok=True)
mapped.to_csv(OUT, index=False, encoding="utf-8")
print(f"Saved {len(mapped)} rows to {OUT.resolve()}")
print(f"Columns: {list(mapped.columns)}")
print(f"File size: {OUT.stat().st_size:,} bytes")

Saved 7800 rows to C:\Users\ahmad\OneDrive\Desktop\AIE bootcamp\week8_project\CONCIERGE\notebooks\data\clinc150_mapped.csv
Columns: ['text', 'label', 'original_intent', 'split']
File size: 511,669 bytes


In [9]:
# (7) SHA-256 hash of the saved CSV. Pin this in services/modelserver/model_card.yaml
# so the modelserver's boot-time hash check (T148) can verify the training data.
sha = hashlib.sha256(OUT.read_bytes()).hexdigest()
print(f"SHA-256: {sha}")
print()
print("Add to services/modelserver/model_card.yaml as:")
print("  training_data:")
print("    source: clinc150_plus")
print("    csv_path: notebooks/data/clinc150_mapped.csv")
print(f"    sha256: {sha}")

SHA-256: b3d6ec9b0ece4493f7d22d7d8f150acb423e2172f4221fb0837ef17e96c95f27

Add to services/modelserver/model_card.yaml as:
  training_data:
    source: clinc150_plus
    csv_path: notebooks/data/clinc150_mapped.csv
    sha256: b3d6ec9b0ece4493f7d22d7d8f150acb423e2172f4221fb0837ef17e96c95f27


## Next

`02_tfidf_logreg_baseline.ipynb` (T131) loads `data/clinc150_mapped.csv`, builds a
TF-IDF + logistic regression classifier on `split == "train"`, evaluates on
`split == "test"`, and reports macro-F1 against the
`classifier_macro_f1 ≥ 0.80` threshold in `eval_thresholds.yaml`.

If the SHA-256 above changes, the training data has changed; retrain and refresh
the model card before merging.